# 06 - Final evaluation on the test set (Part D)

This notebook:
1. trains the final configuration with 3 seeds per model (mean +/- std - the brief's statistical reliability requirement),
2. evaluates on test with the agent-chosen threshold,
3. writes the confusion matrix, boundary metric, and failure-case gallery.

In [ ]:
import sys, os
import numpy as np
sys.path.insert(0, '/content/drive/MyDrive/coco_seg/project/code')
import config as C
import data, engine, losses, models, agent_search, report_utils
C.ensure_dirs()
print('GPU:', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')
import torch
frames = data.load_frames()
CW = report_utils.load_result('class_weights')['median_frequency_weights']
print('frozen choices:')
for name in C.MODELS:
    wc = report_utils.load_result(f'agent_weight_choice_{name}')
    tc = report_utils.load_result(f'agent_threshold_choice_{name}')
    assert wc and tc, 'Run notebook 04 first'
    print(' ', name, '| weights:', wc['chosen_weights'], '| threshold:', tc['chosen_threshold'])

### Train + test, 3 seeds per model

In [ ]:
import metrics as M
for name in C.MODELS:
    existing = report_utils.load_result(f'test_final_{name}')
    if existing and existing.get('complete'):
        print(f'[guard] test_final_{name} already exists - test set is evaluated once. Skipping.')
        continue
    tc = report_utils.load_result(f'agent_threshold_choice_{name}')
    wc = report_utils.load_result(f'agent_weight_choice_{name}')
    per_seed = []
    for seed in C.SEEDS:
        train_loader = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name], augment=True)
        val_loader   = data.get_loader(frames['val'],   'val',   batch_size=C.BATCH_SIZE[name])
        test_loader  = data.get_loader(frames['test'],  'test',  batch_size=C.BATCH_SIZE[name])
        model = models.build_model(name)
        comps = {k: losses.build_loss(k, C.NUM_CLASSES, CW if k in ('ce', 'weighted_ce') else None)
                 for k in wc['components']}
        comp = losses.CompositeLoss(comps, wc['chosen_weights'])
        engine.train(model, f'{name}_final_seed{seed}', train_loader, val_loader,
                     comp, epochs=C.EPOCHS_MAIN, seed=seed)
        test_metrics = engine.evaluate(model, test_loader, threshold=tc['chosen_threshold'], tta=True)
        test_metrics['seed'] = seed
        per_seed.append(test_metrics)
        report_utils.save_result(f'test_final_{name}',   # partial progress, marked incomplete
                                 {'model': name, 'per_seed': per_seed, 'complete': False})
        del model
        torch.cuda.empty_cache()
    keys = ['miou', 'mean_dice', 'mean_precision', 'mean_recall', 'pixel_accuracy', 'boundary_f1']
    agg = {k: report_utils.mean_std_over_seeds(per_seed, k) for k in keys}
    report_utils.save_result(f'test_final_{name}',
                             {'model': name, 'per_seed': per_seed, 'aggregate': agg, 'complete': True})
    print(f'=== {name} TEST (mean over {len(per_seed)} seeds) ===')
    for k in keys:
        print(f'  {k:15s} {agg[k]["mean"]:.4f} +/- {agg[k]["std"]:.4f}')

### Confusion matrix and error analysis

In [ ]:
import numpy as np, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import metrics as M

best = max(C.MODELS, key=lambda m: report_utils.load_result(f'test_final_{m}')['aggregate']['miou']['mean'])
print('best model on test:', best)
tc = report_utils.load_result(f'agent_threshold_choice_{best}')
model = models.build_model(best)
# reuse the seed-42 final weights for the visual analysis
model.load_state_dict(torch.load(engine.ckpt_path(f'{best}_final_seed42'), map_location='cpu')['model'])
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(dev).eval()
test_loader = data.get_loader(frames['test'], 'test', batch_size=C.BATCH_SIZE[best])
cm = np.zeros((C.NUM_CLASSES, C.NUM_CLASSES), dtype=np.int64)
for x, y in test_loader:
    with torch.no_grad():
        probs = torch.softmax(model(x.to(dev)).float(), dim=1)
    conf, pred = probs.max(dim=1)
    pred = torch.where(conf < tc['chosen_threshold'], torch.zeros_like(pred), pred).cpu().numpy()
    for i in range(pred.shape[0]):
        cm += M.confusion_matrix(pred[i], y.numpy()[i], C.NUM_CLASSES)

cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cmn, cmap='Blues')
ax.set_xticks(range(C.NUM_CLASSES), C.CLASS_NAMES, rotation=90)
ax.set_yticks(range(C.NUM_CLASSES), C.CLASS_NAMES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'Pixel confusion matrix (test, {best})')
fig.tight_layout(); fig.savefig(os.path.join(C.FIGURES_DIR, 'confusion_matrix.png'), dpi=140)
plt.close(fig)

# most confused class pairs (off-diagonal, excluding background row/col dominance)
pairs = []
for i in range(C.NUM_CLASSES):
    for j in range(C.NUM_CLASSES):
        if i != j and cmn[i, j] > 0:
            pairs.append((float(cmn[i, j]), C.CLASS_NAMES[i], C.CLASS_NAMES[j]))
pairs.sort(reverse=True)
per_class = M.metrics_from_confusion(cm)
report_utils.save_result('error_analysis', {'best_model': best,
    'confusion_matrix_normalised': cmn.tolist(), 'top_confused_pairs': pairs[:10],
    'per_class_iou_test': per_class['per_class_iou']})
print('top confused pairs (true -> predicted):')
for frac, a, b in pairs[:8]:
    print(f'  {a:12s} -> {b:12s} {frac:.1%}')

### Failure-case gallery (8 worst test images)

In [ ]:
paths = engine.save_worst_overlays(model, test_loader, C.CLASS_NAMES,
                                   os.path.join(C.FIGURES_DIR, 'failures'), k=8)
print('saved', len(paths), 'failure overlays - look at them and describe WHAT fails and WHY in the report')